# Формирование обучающей и тестовой выборок

## Импортирование необходимых библиотек

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

import joblib

## Настройка путей к файлам

In [ ]:
PROCESSED_PATH = Path("../data/processed/cpi_monthly.csv")

OUTPUT_DIR = Path("../data/processed")

## Настройка гиперпараметров

In [ ]:
WINDOW_SIZE = 12
TEST_SIZE = 0.2
RANDOM_STATE = 42

## Загрузка данных

In [ ]:
df = pd.read_csv(PROCESSED_PATH, parse_dates=["Date"])
print(f"Исходный ряд: {df.shape[0]} наблюдений")
display(df.head(3))

## Определение функции для генерации скользящих окон и разбиения выборки

In [ ]:
def prepare_data(df, window_size, split_mode='random', test_size=0.2, random_state=42):
    values = df['cpi_mom'].values.reshape(-1, 1)
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(values)
    
    X, y = [], []
    for i in range(window_size, len(scaled_data)):
        X.append(scaled_data[i-window_size:i, 0])
        y.append(scaled_data[i, 0])
        
    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))
    
    if split_mode == 'random':
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, shuffle=True
        )
    elif split_mode == 'chronological':
        split_idx = int(len(X) * (1 - test_size))
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]
    else:
        raise ValueError("split_mode должен быть 'random' или 'chronological'")
        
    return X_train, y_train, X_test, y_test, scaler

## Случайное разбиение для воспроизведения методологии автора статьи

In [ ]:
X_train_r1, y_train_r1, X_test_r1, y_test_r1, scaler_r1 = prepare_data(
    df, 
    window_size=WINDOW_SIZE, 
    split_mode='random',
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"X_train: {X_train_r1.shape}")
print(f"y_train: {y_train_r1.shape}")
print(f"X_test:  {X_test_r1.shape}")
print(f"y_test:  {y_test_r1.shape}")

## Сохранение тензоров и обученного нормализатора

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUTPUT_DIR / "X_train_r1.npy", X_train_r1)
np.save(OUTPUT_DIR / "y_train_r1.npy", y_train_r1)
np.save(OUTPUT_DIR / "X_test_r1.npy", X_test_r1)
np.save(OUTPUT_DIR / "y_test_r1.npy", y_test_r1)

joblib.dump(scaler_r1, OUTPUT_DIR / "scaler_r1.joblib")

print("Файлы R1 успешно сохранены.")